In [7]:
%load_ext autoreload
%autoreload 2

import IPython
from pathlib import Path
import os
locals = IPython.extract_module_locals() # type: ignore
notebook_name = "/".join(locals[1]["__vsc_ipynb_file__"].split("/"))
os.chdir(Path(notebook_name).parent.parent)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [9]:
spark = SparkSession.builder.appName("Raw").master("local[*]").getOrCreate()


In [10]:
spark

In [11]:
spark.conf.set('spark.sql.caseSensitive', True)

In [12]:
books = spark.read.json(".datalake/raw/amazon_books/Books.jsonl")

In [21]:
metadata = spark.read.json(".datalake/raw/amazon_books/meta_Books.jsonl")

In [22]:
metadata.show()

+--------------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+-----------+-----+-------------+--------------------+--------------------+--------------------+------+
|              author|average_rating|bought_together|          categories|         description|             details|            features|              images|main_category|parent_asin|price|rating_number|               store|            subtitle|               title|videos|
+--------------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+-----------+-----+-------------+--------------------+--------------------+--------------------+------+
|{[Peter Ackroyd, ...|           4.5|           null|[Books, Literatur...|                  []|{null, null, null...|                  []|[{null, https://m...|        Books| 07

In [16]:
books.show()

+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|              images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B09BGPFTDB|           0|[{IMAGE, https://...| B09BGPFTDB|   1.0|It is definitely ...|1642399598485|Not a watercolor ...|AFKZENTNBQ7A7V7UX...|             true|
|0593235657|           1|                  []| 0593235657|   5.0|Updated: after fi...|1640629604904|Updated: after 1s...|AFKZENTNBQ7A7V7UX...|             true|
|1782490671|           0|                  []| 1782490671|   5.0|I bought it for t...|1640383495102|Excellent! I love...|AFKZENTNBQ7A7V7UX...|             true|
|0593138228|           0|         

In [44]:
books = books.withColumn("timestampMonthYear", F.from_unixtime(F.col("timestamp") / 1000, format="yyyy-MM"))

In [45]:
max_timestamp = books.agg(F.max("timestampMonthYear").alias("timestampMonthYear"))

In [46]:
max_timestamp.show()

+------------------+
|timestampMonthYear|
+------------------+
|           2023-09|
+------------------+



In [59]:
ts = max_timestamp.withColumn("date_minus_3_months", F.date_format(F.add_months(F.col("timestampMonthYear"), -3), "yyyy-MM")).select("date_minus_3_months").first()

In [60]:
ts[0]

'2023-06'

In [63]:
books.count()

29475453

In [66]:
books_filtered = books.filter(F.col("timestampMonthYear") >= ts[0])

In [67]:
books_filtered.select("timestampMonthYear").distinct().show()

+------------------+
|timestampMonthYear|
+------------------+
|           2023-06|
|           2023-07|
|           2023-08|
|           2023-09|
+------------------+



In [76]:
books_filtered.count()


77383

In [73]:
items_distinct = books_filtered.select("parent_asin").distinct()

In [74]:
items_distinct.show()

+-----------+
|parent_asin|
+-----------+
| B00SQAR0Q8|
| 1496716914|
| 0134801466|
| B0BZBQYFC2|
| B0C6BXH8GN|
| B0C29YYZ66|
| B0C6C7B6HW|
| 0762745053|
| B0BW2QM63S|
| B09ZWB967R|
| B0BD4H775L|
| 1735564680|
| 1959347144|
| B00684MXX4|
| B08N5YFL78|
| B0C3WMJFWB|
| B093GWDR1L|
| 1250621992|
| B0B7QT3VYZ|
| B09WRJJKXC|
+-----------+
only showing top 20 rows



In [75]:
items_distinct.count()

53007

In [70]:
metadata.count()

4448181

In [69]:
metadata.show()

+--------------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+-----------+-----+-------------+--------------------+--------------------+--------------------+------+
|              author|average_rating|bought_together|          categories|         description|             details|            features|              images|main_category|parent_asin|price|rating_number|               store|            subtitle|               title|videos|
+--------------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+-----------+-----+-------------+--------------------+--------------------+--------------------+------+
|{[Peter Ackroyd, ...|           4.5|           null|[Books, Literatur...|                  []|{null, null, null...|                  []|[{null, https://m...|        Books| 07

In [79]:
metadata_joined = metadata.join(items_distinct, on=["parent_asin"], how="right")

In [80]:
metadata_joined.show()

+-----------+--------------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+----------+-------------+--------------------+--------------------+--------------------+--------------------+
|parent_asin|              author|average_rating|bought_together|          categories|         description|             details|            features|              images|main_category|     price|rating_number|               store|            subtitle|               title|              videos|
+-----------+--------------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+----------+-------------+--------------------+--------------------+--------------------+--------------------+
| 0063278537|                null|           4.7|           null|[Books, Children'...|[About the Author...|{null, null

In [81]:
metadata_joined.count()

53007

In [84]:
ts[0]

'2023-06'

In [13]:
books

DataFrame[asin: string, helpful_vote: bigint, images: array<struct<attachment_type:string,large_image_url:string,medium_image_url:string,small_image_url:string>>, parent_asin: string, rating: double, text: string, timestamp: bigint, title: string, user_id: string, verified_purchase: boolean]

In [85]:
metadata_joined.write.json(f".datalake/raw/amazon_books_from_{ts[0]}/meta_Books.jsonl")

In [18]:
books.write.mode("overwrite").json(f".datalake/raw/amazon_books_from_2023-06/Books.jsonl")

In [ ]:
F.date_format(F.col("timestamp"), "yyyy-MM")